In [ ]:
#remember to delete this one and import correct notebook from kaggle 
import os, json, glob
import numpy as np

SEQUENCE_LENGTH = 20          
KEYPOINTS = 21 * 3
NO_SIGN = "NOSIGN"

candidates = glob.glob('/kaggle/input/*/MP_Data', recursive=True)
if not candidates:
    raise SystemExit("MP_Data not found. Click 'Add Input' and attach your dataset.")
DATA_PATH = candidates[0]
print("Using data at:", DATA_PATH)

folders = os.listdir(DATA_PATH)
letters = sorted(d for d in folders if len(d) == 1 and d.isalpha()
                 and os.path.isdir(os.path.join(DATA_PATH, d)))
actions = letters + ([NO_SIGN] if os.path.isdir(os.path.join(DATA_PATH, NO_SIGN)) else [])
print(len(actions), "classes:", " ".join(actions))
if NO_SIGN not in actions:
    print("Note: no NOSIGN folder - the model will always pick some letter when a hand is visible.")


In [ ]:
# Load every sequence. Still-image sequences have 30 identical frames -> only the first 20 are used.
label_map = {a: i for i, a in enumerate(actions)}
sequences, labels = [], []
skipped = {}

for action in actions:
    for seq in sorted(os.listdir(os.path.join(DATA_PATH, action)), key=int):
        window = []
        for f in range(SEQUENCE_LENGTH):
            p = os.path.join(DATA_PATH, action, seq, f"{f}.npy")
            window.append(np.load(p) if os.path.exists(p) else np.zeros(KEYPOINTS))
        window = np.array(window)
        if not window.any():                      # no hand found at all
            skipped[action] = skipped.get(action, 0) + 1
            continue
        sequences.append(window)
        labels.append(label_map[action])

X = np.array(sequences, dtype=np.float32)
y_int = np.array(labels)
counts = np.bincount(y_int, minlength=len(actions))
print("X shape:", X.shape)
print("Sequences per class:", dict(zip(actions, counts.tolist())))
print("Skipped (no hand):", skipped if skipped else "none")

# Sanity check: J and Z must contain real movement. Still-image data has none.
for letter in ("J", "Z"):
    if letter in label_map:
        motion = X[y_int == label_map[letter]].std(axis=1).mean()
        print(f"{letter}: average movement {motion:.4f}", end="  ")
        print("<- WARNING: looks like still-image data, record clips with record_clips.py"
              if motion < 0.003 else "(ok)")
low = [a for a, c in zip(actions, counts) if c < 40]
if low:
    print("WARNING - fewer than 40 usable sequences for:", " ".join(low))


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y = to_categorical(y_int, num_classes=len(actions))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y_int, random_state=42)

# Augmentation (training data only): tiny per-frame jitter plus a small shift of the whole
# sequence. Makes the still-image sequences look like live frames and adds variety.
rng = np.random.default_rng(0)
def augment(seqs):
    out = seqs.copy()
    present = seqs.any(axis=2, keepdims=True)              # frames where a hand exists
    out += rng.normal(0, 0.004, size=out.shape).astype(np.float32)
    shift = rng.normal(0, 0.02, size=(len(out), 1, 2)).astype(np.float32)
    out[:, :, 0::3] += shift[:, :, 0:1]                    # x
    out[:, :, 1::3] += shift[:, :, 1:2]                    # y
    return np.where(present, out, 0).astype(np.float32)

X_train = np.concatenate([X_train, augment(X_train), augment(X_train)])
y_train = np.concatenate([y_train, y_train, y_train])

# Time-shift augmentation for the motion letters: live, the movement can sit anywhere in the
# 20-frame window. Shifting a clip earlier/later (repeating its first/last frame) teaches that.
def time_shift(seq, s):
    if s > 0:
        return np.concatenate([seq[s:], np.repeat(seq[-1:], s, axis=0)])
    if s < 0:
        return np.concatenate([np.repeat(seq[:1], -s, axis=0), seq[:s]])
    return seq

motion_ids = [label_map[a] for a in ("J", "Z") if a in label_map]
motion_idx = np.where(np.isin(np.argmax(y_train, axis=1), motion_ids))[0]
if len(motion_idx):
    shifted = np.array([time_shift(X_train[i], int(rng.integers(-7, 8)))
                        for i in np.tile(motion_idx, 3)], dtype=np.float32)
    X_train = np.concatenate([X_train, augment(shifted)])
    y_train = np.concatenate([y_train, np.tile(y_train[motion_idx], (3, 1))])
    print("added", len(shifted), "time-shifted J/Z sequences")
print("train:", X_train.shape, " test:", X_test.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, KEYPOINTS)),
    LSTM(64, return_sequences=True),
    LSTM(128, return_sequences=True),
    LSTM(64, return_sequences=False),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(len(actions), activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy",
              metrics=["categorical_accuracy"])
model.summary()


In [ ]:
os.makedirs("/kaggle/working", exist_ok=True)
callbacks = [
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
    ModelCheckpoint("/kaggle/working/best.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(X_train, y_train, epochs=300, batch_size=32,
                    validation_data=(X_test, y_test), callbacks=callbacks)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {acc*100:.1f}%\n")

pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
true = np.argmax(y_test, axis=1)
print(classification_report(true, pred, labels=range(len(actions)),
                            target_names=actions, zero_division=0))

cm = confusion_matrix(true, pred, labels=range(len(actions)))
confused = [(actions[i], actions[j], int(cm[i, j]))
            for i in range(len(actions)) for j in range(len(actions))
            if i != j and cm[i, j] > 0]
confused.sort(key=lambda t: -t[2])
print("Most confused (true, predicted, count):", confused[:10])


In [ ]:
# Save everything the apps need
model.save("/kaggle/working/model.keras")
with open("/kaggle/working/actions.json", "w") as f:
    json.dump(actions, f)

# Weights for the NumPy apps (asl_model.py)
arrays, li, di = {}, 0, 0
for layer in model.layers:
    kind = layer.__class__.__name__
    if kind == "LSTM":
        k, r, b = layer.get_weights()
        arrays[f"lstm{li}_kernel"], arrays[f"lstm{li}_recurrent"], arrays[f"lstm{li}_bias"] = k, r, b
        li += 1
    elif kind == "Dense":
        k, b = layer.get_weights()
        arrays[f"dense{di}_kernel"], arrays[f"dense{di}_bias"] = k, b
        di += 1
assert li == 3 and di == 3, (li, di)
np.savez("/kaggle/working/model_weights.npz", **arrays)
print(os.listdir("/kaggle/working"))
